In [ ]:
"""
LLM Client with Caching

This module provides an LLM client that caches responses to prevent
repeated API calls during testing.
"""

import os
import json
import hashlib
from typing import Optional
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


def _get_cache_file(input_hash: str) -> str:
    """Get cache file path based on input content hash for persistence across runs."""
    cache_dir = os.path.join(os.getcwd(), ".pytest_cache")
    os.makedirs(cache_dir, exist_ok=True)
    return os.path.join(cache_dir, f"cache_{input_hash[:8]}.json")


def _get_input_hash(prompt: str, model: str, max_tokens: int) -> str:
    """Create a hash of input parameters for caching."""
    input_str = f"{prompt}|{model}|{max_tokens}"
    return hashlib.sha256(input_str.encode()).hexdigest()


class LLMClient:
    """Simple LLM client for text completions with caching"""

    def __init__(self):
        """Initialize LLM client with environment validation"""
        self._validate_environment()
        self._client = None

    def _validate_environment(self):
        """Validate required environment variables"""
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError(
                "OpenAI API key is required for LLM operations. "
                "Please set environment variables:\n"
                "OPENAI_API_KEY=your_api_key\n"
                "OPENAI_API_BASE=https://api.openai.com/v1 (optional)\n"
                "Example: export OPENAI_API_KEY=sk-your-key"
            )

    @property
    def client(self) -> OpenAI:
        """Get OpenAI client instance (lazy initialization)"""
        if self._client is None:
            api_key = os.getenv("OPENAI_API_KEY")
            base_url = os.getenv("OPENAI_API_BASE")

            self._client = OpenAI(
                api_key=api_key,
                base_url=base_url if base_url else None
            )
        return self._client

    def complete(self, prompt: str, model: str = "gpt-4o-mini", max_tokens: int = 500) -> str:
        """
        Complete a text prompt with caching

        Args:
            prompt: The text prompt to complete
            model: Model to use (default: gpt-4o-mini)
            max_tokens: Maximum tokens in response (default: 500)

        Returns:
            The LLM response text
        """
        cache_key = _get_input_hash(prompt, model, max_tokens)
        cache_file = _get_cache_file(cache_key)

        try:
            if os.path.exists(cache_file):
                with open(cache_file, "r", encoding="utf-8") as f:
                    cache_data = json.load(f)
                    if cache_key in cache_data:
                        return cache_data[cache_key]
        except (json.JSONDecodeError, IOError, OSError):
            pass

        try:
            response = self.client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=0.1
            )
            content = response.choices[0].message.content

            try:
                cache_data = {}
                if os.path.exists(cache_file):
                    with open(cache_file, "r", encoding="utf-8") as f:
                        cache_data = json.load(f)

                cache_data[cache_key] = content

                with open(cache_file, "w", encoding="utf-8") as f:
                    json.dump(cache_data, f, indent=2)
            except Exception:
                pass

            return content
        except Exception as e:
            raise Exception(f"LLM completion failed: {e}") from e


_llm_client: Optional[LLMClient] = None


def get_llm_client() -> LLMClient:
    """Get the global LLM client instance (singleton pattern)"""
    global _llm_client
    if _llm_client is None:
        _llm_client = LLMClient()
    return _llm_client


In [ ]:
"""
Task Management Tools

This module provides tools for managing tasks in the agent system.
You need to fix validation bugs and implement the delete_task tool.
"""

from typing import Dict, Optional
import uuid

# In-memory task storage (simplified for challenge)
_tasks_db: Dict[str, Dict] = {}


def generate_task_id() -> str:
    """Generate a unique task ID"""
    return f"task_{uuid.uuid4().hex[:8]}"


def create_task(title: str, priority: str) -> Dict:
    """
    Create a new task with the given title and priority.

    Args:
        title: Task title (required)
        priority: Task priority - must be "low", "medium", or "high"

    Returns:
        Dictionary with task details including id, title, priority, status
    """

    if not isinstance(priority, str) or not priority.strip():
        raise ValueError("Priority must be a non-empty string.")

    if not isinstance(title, str) or not title.strip():
        raise ValueError("Title must be a non-empty string.")

    priority = priority.lower()

    if priority not in ["low", "medium", "high"]:
        raise ValueError("Invalid priority. Must be 'low', 'medium', or 'high'.")

    task_id = generate_task_id()
    task = {
        "id": task_id,
        "title": title,
        "priority": priority,
        "status": "pending"
    }
    _tasks_db[task_id] = task
    return task


def get_task(task_id: str) -> Optional[Dict]:
    """
    Retrieve a task by ID.

    Args:
        task_id: The task ID to retrieve

    Returns:
        Task dictionary if found, None otherwise
    """
    return _tasks_db.get(task_id)


def update_task(task_id: str, updates: Dict) -> Dict:
    """
    Update an existing task with new values.

    Args:
        task_id: The task ID to update
        updates: Dictionary of fields to update (e.g., {"status": "completed"})

    Returns:
        Updated task dictionary
    """
    if task_id not in _tasks_db:
        return {"success": False, "error": "Task not found."}

    task = _tasks_db.get(task_id, None)

    if task is None:
        return {"success": False, "error": "Task not found."}

    task.update(updates)
    return task


def delete_task(task_id: str) -> Dict:
    """
    Delete a task by ID.

    Args:
        task_id: The task ID to delete

    Returns:
        Dictionary with success status and message
    """
    if task_id in _tasks_db:
        del _tasks_db[task_id]
        return {"success": True, "message": "Task deleted successfully"}
    else:
        return {"success": False, "error": "Task not found"}


In [ ]:
"""
Agent State Management

This module manages the agent's state including operation history and task tracking.
You need to fix bugs in state management and implement statistics retrieval.
"""

from typing import Dict, List, Optional
from datetime import datetime


class AgentState:
    """
    Manages agent state including operation history and task tracking.
    """

    def __init__(self):
        """Initialize agent state"""
        self.history: List[Dict] = []
        self.tasks: Dict[str, Dict] = {}
        self.completed_tasks: Dict[str, Dict] = []

    def add_operation(self, operation: Dict) -> None:
        """
        Add an operation to the history.

        Args:
            operation: Dictionary with operation details
                Must contain: "action" (str), "timestamp" (str), "result" (dict)

        """
        if not isinstance(operation, Dict):
            raise ValueError("Operation must be a dictionary.")

        required_fields = ["action", "timestamp", "result"]
        for field in required_fields:
            if field not in operation:
                raise ValueError(f"Operation is missing required field: {field}")

        if not isinstance(operation["result"], dict):
            raise ValueError("Operation 'result' must be a dictionary.")

        self.history.append(operation)

    def mark_task_completed(self, task_id: str) -> None:
        """
        Mark a task as completed.

        Args:
            task_id: The task ID to mark as completed
        """
        if not isinstance(task_id, str) or not task_id:
            raise ValueError("task_id must be a non-empty string")

        if task_id in self.tasks:
            self.tasks[task_id]["status"] = "completed"
        else:
            self.tasks[task_id] = {"id": task_id, "status": "completed"}

        # Record completion metadata
        self.completed_tasks.append(task_id) if task_id not in self.completed_tasks else None

    def get_statistics(self) -> Dict:
        """
        Get statistics about agent operations and tasks.

        Returns:
            Dictionary with statistics:
            - total_operations: Total number of operations performed
            - completed_tasks_count: Number of completed tasks
            - pending_tasks_count: Number of pending tasks
            - high_priority_tasks: Number of high priority tasks
        """
        total_operations = len(self.history)
        completed_tasks_count = len(self.completed_tasks)
        pending_tasks_count = sum(1 for task in self.tasks.values() if task.get("status") != "completed")
        high_priority_tasks = sum(1 for task in self.tasks.values() if task.get("priority") == "high")

        return {
            "total_operations": total_operations,
            "completed_tasks_count": completed_tasks_count,
            "pending_tasks_count": pending_tasks_count,
            "high_priority_tasks": high_priority_tasks
        }

    def get_history(self) -> List[Dict]:
        """Get the full operation history"""
        return self.history.copy()

    def get_tasks(self) -> Dict[str, Dict]:
        """Get all tasks"""
        return self.tasks.copy()

In [ ]:
"""
Task Management Agent

This module implements the agent that uses tools and state management
to handle task management requests.
"""

from typing import Dict, Optional
import json
from datetime import datetime
from tools import create_task, get_task, update_task, delete_task
from state_manager import AgentState
from llm import get_llm_client


class TaskAgent:
    """
    Agent that manages tasks using tools and maintains state.
    """

    def __init__(self, state_manager: Optional[AgentState] = None):
        """
        Initialize the task agent.

        Args:
            state_manager: Optional state manager instance (creates new one if not provided)
        """
        self.state = state_manager or AgentState()
        self.llm = get_llm_client()
        self.tools = {
            "create_task": create_task,
            "get_task": get_task,
            "update_task": update_task,
            "delete_task": delete_task,
        }

    def execute_action(self, action: Dict) -> Dict:
        """
        Execute a tool action.

        Args:
            action: Dictionary with:
                - "tool": Tool name (e.g., "create_task")
                - "params": Dictionary of parameters for the tool

        Returns:
            Dictionary with:
                - "success": Boolean indicating if action succeeded
                - "result": Tool result if successful
                - "error": Error message if failed
        """
        if not isinstance(action, dict):
            return {"success": False, "error": "Invalid action format"}

        required_fields = ["tool", "params"]
        for field in required_fields:
            if field not in action:
                return {"success": False, "error": f"Missing required field: {field}"}

        tool, params = action["tool"], action["params"]
        if tool not in self.tools:
            return {"success": False, "error": f"Unknown tool: {tool}"}

        try:
            result = self.tools[tool](**params)
            return {"success": True, "result": result}
        except Exception as e:
            return {"success": False, "error": str(e)}

    def update_state(self, action: Dict, result: Dict) -> None:
        """
        Update agent state after an action.

        Args:
            action: The action that was executed
            result: The result from execute_action
        """
        if not isinstance(action, dict):
            raise ValueError("Invalid action format")

        if not isinstance(result, dict):
            raise ValueError("Invalid result format")

        # 1. Create operation record and add to history
        operation = {
            "action": action.get("tool", "unknown"),
            "timestamp": datetime.utcnow().isoformat(),
            "result": result,
        }
        self.state.add_operation(operation)

        # 2. If create_task succeeded, add task to state.tasks
        if action.get("tool") == "create_task" and result.get("success"):
            task_id = result["result"].get("id")
            if task_id:
                self.state.tasks[task_id] = result["result"]

        # 3. If update_task succeeded and status is "completed", mark completed
        elif action.get("tool") == "update_task" and result.get("success"):
            task_id = action["params"].get("task_id")
            if task_id and result["result"].get("status") == "completed":
                self.state.mark_task_completed(task_id)

    def run(self, user_request: str) -> Dict:
        """
        Run the agent loop to process a user request.

        Args:
            user_request: User's request as a string

        Returns:
            Dictionary with:
                - "success": Boolean
                - "result": Final result
                - "operations": List of operations performed
        """
        if not isinstance(user_request, str):
            return {"success": False, "error": "Invalid user request format"}

        operations = []
        for _ in range(5):  # Max 5 iterations
            response = self.llm.complete(user_request)

            if not response:
                break

            try:
                parsed = json.loads(response)
            except json.JSONDecodeError:
                break

            action = {"tool": parsed.get("tool"), "params": parsed.get("params", {})}

            result = self.execute_action(action)

            if result.get("success"):
                self.update_state(action, result)
                operations.append({"action": action, "result": result})
            else:
                operations.append({"action": action, "result": result})
                break

            # If LLM indicates done, stop
            if parsed.get("done", False):
                break

        return {
            "success": bool(operations)
            and operations[-1]["result"].get("success", False),
            "result": operations[-1]["result"] if operations else None,
            "operations": operations,
        }
